In [ ]:
import pandas as pd
from datetime import datetime
import supabase

pd.set_option('display.max_columns', None)

from extract import *
from transform import *
from load import *
from extract import supabase


def obter_mes_referencia():

    meses = {
        1: "janeiro",
        2: "fevereiro",
        3: "março",
        4: "abril",
        5: "maio",
        6: "junho",
        7: "julho",
        8: "agosto",
        9: "setembro",
        10: "outubro",
        11: "novembro",
        12: "dezembro"
    }

    hoje = datetime.today()

    if hoje.month == 1:
        mes_num = 12
        ano_ref = hoje.year - 1
    else:
        mes_num = hoje.month - 1
        ano_ref = hoje.year

    return meses[mes_num], ano_ref


def verificar_mes_referencia_people_analytics():

    mes_referencia, ano_ref = obter_mes_referencia()

    resposta = (
        supabase
        .table("people_analytics")
        .select("id")
        .eq("mes_referencia", mes_referencia)
        .limit(1)
        .execute()
    )

    if resposta.data:
        print(
            f"[INFO] Já existem registros para "
            f"{mes_referencia}/{ano_ref}"
        )
        return True

    print(
        f"[INFO] Não existem registros para "
        f"{mes_referencia}/{ano_ref}"
    )

    return False


# =====================================================
# EXECUÇÃO
# =====================================================

if verificar_mes_referencia_people_analytics():

    print("[INFO] Processamento cancelado.")

else:

    print("[INFO] Iniciando processamento...")

    mes_referencia, ano_ref = obter_mes_referencia()

    tabelas_brutas = carregar_tabelas()

    tabelas_transformadas = orquestrar_etl_transformacao(
        tabelas_brutas
    )

    df_score_final = orquestrar_feature_engineering(
        tabelas_brutas['usuarios'],
        tabelas_transformadas
    )

    df_analise_final = calcular_scores_heuristicos(
        df_score_final
    )

    df_analise_final["analise_pa"] = None
    df_analise_final["mes_referencia"] = mes_referencia

    # Ajusta nulos
    df_analise_final = df_analise_final.where(
        pd.notnull(df_analise_final),
        None
    )

    # Ajusta datas
    for col in df_analise_final.columns:

        if pd.api.types.is_datetime64_any_dtype(
            df_analise_final[col]
        ):
            df_analise_final[col] = (
                df_analise_final[col]
                .dt.strftime("%Y-%m-%d")
            )
            
    df_analise_final = df_analise_final.rename(
    columns={"id": "usuario_id"}
    )

    inserir_people_analytics(
        df_analise_final
    )

    print(
        f"[INFO] Carga concluída para "
        f"{mes_referencia}/{ano_ref}"
    )

[INFO] Não existem registros para abril/2026
[INFO] Iniciando processamento...
Tabela 'usuarios' carregada: 12 linhas x 12 colunas
Tabela 'pulse' carregada: 12 linhas x 7 colunas
Tabela 'historico_cursos' carregada: 43 linhas x 9 colunas
Tabela 'historico_promocoes' carregada: 21 linhas x 8 colunas
Tabela 'pontos' carregada: 10180 linhas x 7 colunas
Tabela 'banco_horas' carregada: 12 linhas x 5 colunas
Tabela 'ferias' carregada: 240 linhas x 9 colunas
Tabela 'atestados' carregada: 23 linhas x 9 colunas
Iniciando ETL de People Analytics...
Transformando dados de Ponto...
Mapeando histórico de Atestados...
Processando histórico de Férias...
Calculando tempos de Cargo/Promoção...
Separando métricas de Educação Corporativa...
ETL concluído com sucesso! 

Iniciando construção da Feature Matrix (df_score)...
Feature Matrix construída com sucesso! (12 funcionários x 26 features)
[INFO] 12 registros inseridos em people_analytics
[INFO] Carga concluída para abril/2026


In [ ]:
{"idx":0,"usuario_id":"6373eece-2622-490d-bb64-bd5eca36d049","nome":"Carlos Mendes","created_at":"2026-05-27 01:37:03.32678+00","data_contratacao":"2021-03-15","gerente_geral":"sim","tenant_id":1,"cargo":"Gerente de Tecnologia","gerente_id":null,"nivel_cargo":7,"faltas_ultimo_mes":0,"horas_extras_ultimo_mes":"0.73","qtd_atrasos_ultimo_mes":0,"horas_atraso_ultimo_mes":"0.0","media_atrasos_trimestre":"1.0","ausencias_atestado_ultimo_mes":1,"media_ausencia_atestado_trimestre":"0.33","tempo_meses_ultimas_férias":"11.1","qtt_promocoes":1,"tempo_ultima_promocao":"23.6","cursos_obg_vencidos":0,"cursos_obg_no_prazo":0,"cursos_obg_adiantados":0,"qtt_cursos_opcioanis_2anos":0,"score_humor":50,"sentimento_predominante":"Neutro","resumo_ia":"Sem interações recentes.","score_burnout":0,"score_turnover":0,"score_engajamento":20,"score_elegibilidade_promocao":40,"posicao_burnout":"Normal","posicao_turnover":"Bottom 20% (Saudável/Seguro)","posicao_engajamento":"Normal","posicao_elegibilidade_promocao":"Top 20% (Destaque)","analise_pa":null,"mes_referencia":"marco","id":1}
{"idx":0,"usuario_id":"6373eece-2622-490d-bb64-bd5eca36d049","nome":"Carlos Mendes","created_at":"2026-05-27 00:00:00+00.     ","data_contratacao":"2021-03-15","gerente_geral":"sim","tenant_id":1,"cargo":"Gerente de Tecnologia","gerente_id":null,"nivel_cargo":7,"faltas_ultimo_mes":0,"horas_extras_ultimo_mes":"0.73","qtd_atrasos_ultimo_mes":0,"horas_atraso_ultimo_mes":"0.0","media_atrasos_trimestre":"1.0","ausencias_atestado_ultimo_mes":1,"media_ausencia_atestado_trimestre":"0.33","tempo_meses_ultimas_férias":"11.1","qtt_promocoes":1,"tempo_ultima_promocao":"23.6","cursos_obg_vencidos":0,"cursos_obg_no_prazo":0,"cursos_obg_adiantados":0,"qtt_cursos_opcioanis_2anos":0,"score_humor":50,"sentimento_predominante":"Neutro","resumo_ia":"Sem interações recentes.","score_burnout":0,"score_turnover":0,"score_engajamento":20,"score_elegibilidade_promocao":40,"posicao_burnout":null,"posicao_turnover":null,"posicao_engajamento":null,"posicao_elegibilidade_promocao":null,"analise_pa":null,"mes_referencia":"abril","id":50}

['id', 'nome', 'created_at', 'data_contratacao', 'gerente_geral', 'tenant_id', 'cargo', 'gerente_id', 'nivel_cargo', 'faltas_ultimo_mes', 'horas_extras_ultimo_mes', 'qtd_atrasos_ultimo_mes', 'horas_atraso_ultimo_mes', 'media_atrasos_trimestre', 'ausencias_atestado_ultimo_mes', 'media_ausencia_atestado_trimestre', 'tempo_meses_ultimas_férias', 'qtt_promocoes', 'tempo_ultima_promocao', 'cursos_obg_vencidos', 'cursos_obg_no_prazo', 'cursos_obg_adiantados', 'qtt_cursos_opcioanis_2anos', 'score_humor', 'sentimento_predominante', 'resumo_ia', 'score_burnout', 'score_turnover', 'score_engajamento', 'score_elegibilidade_promocao', 'analise_pa', 'mes_referencia']


In [ ]:
from datetime import datetime

def verificar_mes_referencia_people_analytics():
    
    meses = {
        1: "janeiro",
        2: "fevereiro",
        3: "março",
        4: "abril",
        5: "maio",
        6: "junho",
        7: "julho",
        8: "agosto",
        9: "setembro",
        10: "outubro",
        11: "novembro",
        12: "dezembro"
    }

    hoje = datetime.today()

    if hoje.month == 1:
        mes_num = 12
        ano_ref = hoje.year - 1
    else:
        mes_num = hoje.month - 1
        ano_ref = hoje.year

    mes_referencia = meses[mes_num]

    resposta = (
        supabase
        .table("people_analytics")
        .select("id")
        .eq("mes_referencia", mes_referencia)
        .limit(1)
        .execute()
    )

    if resposta.data:
        print(
            f"Já existem registros para "
            f"{mes_referencia}/{ano_ref}"
        )
        return True

    print(
        f"Não existem registros para "
        f"{mes_referencia}/{ano_ref}"
    )
    return False

if verificar_mes_referencia_people_analytics():
    print("Processamento cancelado.")
else:
    print("Executando carga...")

Já existem registros para abril/2026
Processamento cancelado.
